In [3]:
import os;
from langchain_ollama import OllamaEmbeddings;
from langchain_chroma import Chroma;

persist_directory = os.path.join(os.getcwd(), "db")
embedding = OllamaEmbeddings(model="nomic-embed-text")
docs_dir = os.path.join(os.getcwd(), "docs")
docs_dir

'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs'

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader(os.path.join(docs_dir, "bom.txt"))
document = loader.load()
document

[Document(metadata={'source': 'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product. A BOM may be used for communication between manufacturing partners or confined to a single manufacturing plant. A bill of materials is often tied to a production order whose issuance may generate reservations for components in the bill of materials that are in stock and requisitions for components that are not in stock.\n\nThe first hierarchical databases were developed for automating bills of materials for manufacturing organizations in the early 1960s. At present, this BOM is used as a database to identify the many parts and their codes in automobile manufacturing companies.\n\nA BOM can

For most RAG (Retrieval-Augmented Generation) applications, the standard "sweet spot" for chunk overlap is **10% to 20%** of your chunk size.

In [9]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=10)
chunks = text_splitter.split_documents(document)
chunks

[Document(metadata={'source': 'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product. A BOM may be used for communication between manufacturing partners or confined to a single manufacturing plant. A bill of materials is often tied to a production order whose issuance may generate reservations for components in the bill of materials that are in stock and requisitions for components that are not in stock.\n\nThe first hierarchical databases were developed for automating bills of materials for manufacturing organizations in the early 1960s. At present, this BOM is used as a database to identify the many parts and their codes in automobile manufacturing companies.'),
 Document

Note: Calling **Chroma.from_documents** twice will not replace the original content; it will append the new documents to the existing collection, potentially creating duplicates.

In [10]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory
)

retriever = vectordb.as_retriever(search_kwargs={"k": 2})
relevant_docs = retriever.invoke("Types of BOM")

In [11]:
relevant_docs

[Document(id='dbe0ae0e-9e74-43cf-b2b1-5810d83f33ff', metadata={'source': 'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\bom.txt'}, page_content='A bill of materials "implosion" links component pieces to a major assembly, while a bill of materials "explosion" breaks apart each assembly or sub-assembly into its component parts.\n\nUsage\n=======\nIn process industries, the BOM is also known as the formula, recipe, or ingredients list. The phrase "bill of material" (or "BOM") is frequently used by engineers attributively to refer not to the literal bill, but to the current production configuration of a product, to distinguish it from modified or improved versions under study or in test.\n\nIn electronics, the BOM represents the list of components used on the printed circuit board (PCB). Once the design of the circuit is completed, the BOM list is passed on to the PCB layout engineer as well as the component engineer who will procure the compo

**<h3>Load Existing DB and basic operation</h3>**

In [12]:
vectordb = Chroma(persist_directory=persist_directory, embedding_function=embedding)


In [14]:
# Clears the data but keeps the folder structure
# vectordb.delete_collection()

# Append new data
loader = TextLoader(os.path.join(docs_dir, "einvoice.txt"))
document = loader.load()
chunks = text_splitter.split_documents(document)
vectordb.add_documents(chunks)

['e2ba49ac-f1d3-423c-a4cf-df8ec246a51d',
 'c0f5d1e6-f646-4082-b2e9-d0d6833417f9',
 '591f4426-276f-4453-ac1b-695ebbab45b4',
 '035a2836-b192-4ac2-8689-43b97ecf9b66']

In [19]:
existing_docs = vectordb.get(
    where={"$or": [{"source": os.path.join(docs_dir, "bom.txt")}, {"source": os.path.join(docs_dir, "einvoice.txt")}]},
    limit=3
)

# Get the first document in the database
# sample = vectordb.get(limit=1)
# sample

existing_docs

{'ids': ['ad687a9b-c703-491b-93db-29eaae4458a8',
  '366aceaa-4ae1-48df-8a97-f80963fa3766',
  'dbe0ae0e-9e74-43cf-b2b1-5810d83f33ff'],
 'embeddings': None,
 'documents': ['A bill of materials or product structure (sometimes bill of material, BOM or associated list) is a list of the raw materials, sub-assemblies, intermediate assemblies, sub-components, parts, and the quantities of each needed to manufacture an end product. A BOM may be used for communication between manufacturing partners or confined to a single manufacturing plant. A bill of materials is often tied to a production order whose issuance may generate reservations for components in the bill of materials that are in stock and requisitions for components that are not in stock.\n\nThe first hierarchical databases were developed for automating bills of materials for manufacturing organizations in the early 1960s. At present, this BOM is used as a database to identify the many parts and their codes in automobile manufacturing c

In [20]:
# Delete specific chunks based on their source
# vectordb.delete(where={"source": os.path.join(docs_dir, "bom.txt")})

# Delete specific chunks based on their IDs
# ids_to_delete = [chunk['id'] for chunk in existing_docs]
# vectordb.delete(ids=ids_to_delete)

In [21]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2})
relevant_docs = retriever.invoke("Requirement for einvoice?")
relevant_docs

[Document(id='591f4426-276f-4453-ac1b-695ebbab45b4', metadata={'source': 'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\einvoice.txt'}, page_content='What are the requirements for e-invoicing in Malaysia?\n1. e-Invoices must be generated for B2B, B2C, and B2G transactions in UBL 2.1 format (XML or JSON)\n2. e-Invoices can be generated and submitted either manually via the MyInvois Portal or automatically through direct API integration.\n3. Each e-invoice must include 55 specific data fields, covering seller and buyer details, transaction items, quantities, prices, taxes, totals, and payment information.\n4. All e-invoices must be digitally signed using a Digital Certificate issued by IRBM.\n\ne-Invoicing Process for B2B Transactions'),
 Document(id='e2ba49ac-f1d3-423c-a4cf-df8ec246a51d', metadata={'source': 'c:\\Users\\wengshang.hoo\\Desktop\\pp\\learning-note02\\rag\\1. Basic Rag With Chroma\\docs\\einvoice.txt'}, page_content='e-Invoicin